# CelebA Cost-Aware CLAQ Experiment

This notebook evaluates cost-aware adaptive querying on CelebA using the `Attractive` prediction task. It uses a held-out subset of the training split to calibrate synthetic response-uncertainty costs, screens `lambda_c` with one seed on validation data, and evaluates the selected value over five seeds.

Uniform query-count cost is represented by the `lambda_c = 0` baseline because a constant per-query cost has zero actor gradient. Quantitative test results report accuracy, macro F1, realized acquisition cost, query count, and sensitive-query count.

In [1]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import t
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Subset

from claq.config import CelebAClaqConfig, default_paths
from claq.core import (
    apply_query_distribution,
    build_concept_dictionary,
    build_concept_qa_inputs,
    build_uncertainty_cost,
    build_uniform_cost,
    encode_images,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_run_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from claq.data import (
    get_celeba_concept_qa_loaders,
    get_celeba_datasets,
    load_celeba_attribute_spec,
)
from claq.models import ConceptNet2
from claq.training import HistorySamplingConfig, build_claq_models, fit_concept_qa, fit_claq, seed_everything

In [2]:
repo_root = Path.cwd().resolve()
if not (repo_root / "claq").exists() and (repo_root.parent / "claq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()
runs_dir = paths.runs_root
config = CelebAClaqConfig()
device = config.device

SEEDS = (0, 1, 2, 3, 4)
SCREENING_SEED = SEEDS[0]
CALIBRATION_SEED = 1729
CALIBRATION_SIZE = 2_000
MAX_QUESTIONS = 20
TRAIN_ROLLOUT_STEPS = MAX_QUESTIONS
NUM_EPOCHS = config.default_train_epochs  # match the original CelebA experiment
DESIGNATED_QUERY_WEIGHT = 1.0
SENSITIVE_WEIGHT = 0.0
LAMBDA_C_CANDIDATES = (0.01, 0.03, 0.1, 0.2, 0.3)
MAX_VALIDATION_ACCURACY_DROP = 0.01
COST_BUDGETS = (4, 8, 12, 16, 20)

figure_ext = ".svg"
download_celeba = False
experiment_name = "celeba_attractive_cost"
qa_experiment_name = "celeba_attractive"
train_min_history = 0
train_max_history = 16
actor_eps_end = 0.2
actor_eps_anneal_epochs = config.default_train_epochs

concept_qa_max_train_batches = 80 if device.type == "cpu" else None
concept_qa_max_eval_batches = 20 if device.type == "cpu" else None
claq_max_train_batches = 80 if device.type == "cpu" else None
claq_max_eval_batches = 20 if device.type == "cpu" else None
seed_everything(SCREENING_SEED)

print({"device": str(device), "seeds": SEEDS, "calibration_size": CALIBRATION_SIZE})

{'device': 'cuda', 'seeds': (0, 1, 2, 3, 4), 'calibration_size': 2000}


In [3]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
spec = load_celeba_attribute_spec(
    root=paths.data_root,
    target_attribute=config.target_attribute,
    sensitive_attributes=config.sensitive_attributes,
    download=download_celeba,
)
concepts = spec.concept_names
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sens_idx = spec.sensitive_indices
sensitive_mask = make_sensitive_mask(len(concepts), sens_idx, device)
sample_sensitive_attribute = "Male"
sample_sens_idx = torch.tensor(
    [spec.query_attribute_names.index(sample_sensitive_attribute)], dtype=torch.long
)

qa_train_loader, qa_valid_loader = get_celeba_concept_qa_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    download=download_celeba,
)
train_dataset, validation_dataset, test_dataset = get_celeba_datasets(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    return_query_targets=True,
    download=download_celeba,
)
if CALIBRATION_SIZE >= len(train_dataset):
    raise ValueError("CALIBRATION_SIZE must be smaller than the CelebA training split")

split_generator = torch.Generator().manual_seed(CALIBRATION_SEED)
permutation = torch.randperm(len(train_dataset), generator=split_generator).tolist()
calibration_indices = permutation[:CALIBRATION_SIZE]
policy_train_indices = permutation[CALIBRATION_SIZE:]

policy_train_loader = DataLoader(
    Subset(train_dataset, policy_train_indices),
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
)
calibration_loader = DataLoader(
    Subset(train_dataset, calibration_indices),
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
)

print({
    "target": spec.target_attribute,
    "queries": len(concepts),
    "policy_train": len(policy_train_indices),
    "calibration": len(calibration_indices),
    "validation": len(validation_dataset),
    "test": len(test_dataset),
    "sensitive_queries": spec.sensitive_attribute_names,
})

{'target': 'Attractive', 'queries': 39, 'policy_train': 160770, 'calibration': 2000, 'validation': 19867, 'test': 19962, 'sensitive_queries': ['Male', 'No_Beard', 'Mustache', 'Goatee', 'Sideburns', '5_o_Clock_Shadow', 'Heavy_Makeup', 'Wearing_Lipstick']}


In [4]:
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_experiment_name}.pt"
qa_history_path = runs_dir / f"concept_qa_{qa_experiment_name}_history.json"

if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
else:
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.Adam(qa_model.parameters(), lr=config.learning_rate)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        qa_optimizer, T_max=max(config.concept_qa_epochs, 1)
    )
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=qa_train_loader,
        eval_loader=qa_valid_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=config.concept_qa_epochs,
        model_clip=model_clip,
        dictionary=dictionary,
        class_concept_targets=None,
        clip_device=device,
        train_device=device,
        max_train_batches=concept_qa_max_train_batches,
        max_eval_batches=concept_qa_max_eval_batches,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(qa_history_path, "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()

print(f"Concept-QA checkpoint: {qa_checkpoint}")

Concept-QA checkpoint: /home/jupyter/claq/artifacts/models/concept_qa_celeba_attractive.pt


In [5]:
@torch.no_grad()
def soft_response_probabilities(loader, qa_chunk=4096):
    answering_model.eval()
    probabilities = []
    for images, _labels, _query_targets in loader:
        image_features = encode_images(model_clip=model_clip, images=images, device=device)
        qa_inputs = build_concept_qa_inputs(
            image_features=image_features,
            dictionary=dictionary,
        ).to(next(answering_model.parameters()).device).float()
        logits = []
        for start in range(0, qa_inputs.size(0), qa_chunk):
            logits.append(answering_model(qa_inputs[start : start + qa_chunk]))
        batch_logits = torch.cat(logits).view(images.size(0), len(concepts))
        probabilities.append(torch.sigmoid(batch_logits).cpu())
    return torch.cat(probabilities, dim=0)


calibration_soft = soft_response_probabilities(calibration_loader)
# Keep designated sensitive queries at unit acquisition cost, as in the BiB
# protocol; their use is controlled separately by lambda_q.
learned_query_mask = 1.0 - sensitive_mask.cpu()
uniform_cost = build_uniform_cost(len(concepts), device=device)
uncertainty_cost = build_uncertainty_cost(
    calibration_soft,
    learned_mask=learned_query_mask,
    device=device,
)
cost_table = pd.DataFrame({
    "query_index": np.arange(len(concepts)),
    "attribute": spec.query_attribute_names,
    "concept": concepts,
    "sensitive": sensitive_mask.cpu().numpy().astype(bool),
    "uniform_cost": uniform_cost.cpu().numpy(),
    "uncertainty_cost": uncertainty_cost.cpu().numpy(),
})

display(cost_table.sort_values("uncertainty_cost", ascending=False).head(10))

,query_index,attribute,concept,sensitive,uniform_cost,uncertainty_cost
24,24,Oval_Face,oval face,False,1.0,1.894180
26,26,Pointy_Nose,pointy nose,False,1.0,1.866068
5,5,Big_Lips,big lips,False,1.0,1.847866
31,31,Straight_Hair,straight hair,False,1.0,1.802759
2,2,Bags_Under_Eyes,bags under eyes,False,1.0,1.783754
6,6,Big_Nose,big nose,False,1.0,1.774442
1,1,Arched_Eyebrows,arched eyebrows,False,1.0,1.771436
10,10,Brown_Hair,brown hair,False,1.0,1.768258
36,36,Wearing_Necklace,wearing necklace,False,1.0,1.749686
32,32,Wavy_Hair,wavy hair,False,1.0,1.734621


## Cost-aware policy training

Each example is unrolled for `TRAIN_ROLLOUT_STEPS` queries. The task loss is averaged over all rollout prefixes, the cumulative acquisition cost is weighted by `lambda_c`, and the designated sensitive-query count is weighted by the fixed `DESIGNATED_QUERY_WEIGHT`. The latter is held constant across all conditions.

In [6]:
def fit_cost_policy(run_name, lambda_c, seed, force_retrain=False):
    seed_everything(seed)
    run_stem = f"{experiment_name}_{run_name}_seed_{seed}"
    checkpoint_path = paths.checkpoints_root / f"{run_stem}_best.pt"
    history_path = runs_dir / f"{run_stem}_history.csv"

    if checkpoint_path.exists() and history_path.exists() and not force_retrain:
        bundle = load_run_bundle(
            checkpoint_path,
            device=device,
            max_queries=len(concepts),
            num_classes=config.num_classes,
            actor_eps=config.actor_eps,
        )
        history = pd.read_csv(history_path)
        bundle.update({
            "run_name": run_name,
            "seed": seed,
            "lambda_c": lambda_c,
            "lambda_q": DESIGNATED_QUERY_WEIGHT,
            "history": history,
            "best_epoch": int(bundle["meta"]["best_epoch"]),
        })
        return bundle

    actor, classifier, s_head = build_claq_models(
        max_queries=len(concepts),
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=config.learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=train_min_history,
        max_history=train_max_history,
        non_sensitive_only=False,
    )
    history_rows, best = fit_claq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=policy_train_loader,
        test_loader=validation_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_s=SENSITIVE_WEIGHT,
        lambda_c=lambda_c,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=NUM_EPOCHS,
        max_train_batches=claq_max_train_batches,
        max_test_batches=claq_max_eval_batches,
        actor_eps_end=actor_eps_end,
        actor_eps_anneal_epochs=actor_eps_anneal_epochs,
        sensitive_target_mode="max",
        sensitive_target_indices=sample_sens_idx,
        cost_vector=uncertainty_cost,
        training_rollout_steps=TRAIN_ROLLOUT_STEPS,
        lambda_q=DESIGNATED_QUERY_WEIGHT,
        designated_query_mask=sensitive_mask,
    )
    history = pd.DataFrame(history_rows).assign(run_name=run_name, seed=seed)
    history.to_csv(history_path, index=False)
    save_bundle_checkpoint(
        checkpoint_path=checkpoint_path,
        metadata={
            "run_name": run_name,
            "seed": seed,
            "lambda_c": lambda_c,
            "lambda_q": DESIGNATED_QUERY_WEIGHT,
            "best_epoch": best["epoch"],
            "best_validation_accuracy": best["test_acc"],
            "actor_state_dict": best["actor_state_dict"],
            "classifier_state_dict": best["classifier_state_dict"],
            "s_head_state_dict": best["s_head_state_dict"],
        },
    )
    bundle = load_run_bundle(
        checkpoint_path,
        device=device,
        max_queries=len(concepts),
        num_classes=config.num_classes,
        actor_eps=config.actor_eps,
    )
    bundle.update({
        "run_name": run_name,
        "seed": seed,
        "lambda_c": lambda_c,
        "lambda_q": DESIGNATED_QUERY_WEIGHT,
        "history": history,
        "best_epoch": best["epoch"],
    })
    return bundle

In [7]:
@torch.no_grad()
def materialize_answers(loader, qa_chunk=4096):
    answering_model.eval()
    answers_all, labels_all = [], []
    for images, labels, _query_targets in loader:
        image_features = encode_images(model_clip=model_clip, images=images, device=device)
        qa_inputs = build_concept_qa_inputs(
            image_features=image_features,
            dictionary=dictionary,
        ).to(next(answering_model.parameters()).device).float()
        logits = []
        for start in range(0, qa_inputs.size(0), qa_chunk):
            logits.append(answering_model(qa_inputs[start : start + qa_chunk]))
        batch_logits = torch.cat(logits).view(images.size(0), len(concepts))
        answers = torch.where(
            batch_logits > config.threshold_for_binarization,
            torch.ones_like(batch_logits),
            -torch.ones_like(batch_logits),
        )
        answers_all.append(answers.cpu())
        labels_all.append(labels.cpu())
    return torch.cat(answers_all), torch.cat(labels_all)


validation_answers, validation_labels = materialize_answers(validation_loader)
test_answers, test_labels = materialize_answers(test_loader)
print({"validation_answers": tuple(validation_answers.shape), "test_answers": tuple(test_answers.shape)})


@torch.no_grad()
def rollout_curve(run, answers_cpu, labels_cpu, meter_cost, batch_size=config.batch_size):
    run["actor"].eval()
    run["classifier"].eval()
    predictions_by_step = [[] for _ in range(MAX_QUESTIONS + 1)]
    costs_by_step = [[] for _ in range(MAX_QUESTIONS + 1)]
    sensitive_by_step = [[] for _ in range(MAX_QUESTIONS + 1)]

    for start in range(0, len(answers_cpu), batch_size):
        answers = answers_cpu[start : start + batch_size].to(device)
        mask = torch.zeros_like(answers)
        revealed = torch.zeros_like(answers)
        cumulative_cost = torch.zeros(answers.size(0), device=device)
        cumulative_sensitive = torch.zeros(answers.size(0), device=device)

        for step in range(MAX_QUESTIONS + 1):
            predictions_by_step[step].extend(
                run["classifier"](revealed).argmax(dim=1).cpu().tolist()
            )
            costs_by_step[step].extend(cumulative_cost.cpu().tolist())
            sensitive_by_step[step].extend(cumulative_sensitive.cpu().tolist())
            if step == MAX_QUESTIONS:
                break
            query_distribution = run["actor"](revealed, mask)
            cumulative_cost += (query_distribution * meter_cost).sum(dim=1)
            cumulative_sensitive += (query_distribution * sensitive_mask).sum(dim=1)
            revealed = apply_query_distribution(revealed, answers, query_distribution)
            mask = torch.clamp(mask + query_distribution.detach(), 0.0, 1.0)

    labels = labels_cpu.tolist()
    return pd.DataFrame([
        {
            "run_name": run["run_name"],
            "seed": run["seed"],
            "step": step,
            "accuracy": accuracy_score(labels, predictions_by_step[step]),
            "macro_f1": f1_score(labels, predictions_by_step[step], average="macro"),
            "mean_cost": float(np.mean(costs_by_step[step])),
            "mean_sensitive_queries": float(np.mean(sensitive_by_step[step])),
        }
        for step in range(MAX_QUESTIONS + 1)
    ])


def summarize_curve(curve):
    return curve

{'validation_answers': (19867, 39), 'test_answers': (19962, 39)}


## Validation screening and multi-seed evaluation

The test split is not used to choose `lambda_c`. Candidate values are screened with one seed, selecting the lowest validation cost among candidates whose accuracy is within one percentage point of the baseline. Only the selected value is then trained for the remaining seeds.

In [ ]:
def relabel_run(run, run_name):
    return {**run, "run_name": run_name}


baseline_screening = fit_cost_policy("baseline", lambda_c=0.0, seed=SCREENING_SEED)
screening_runs = {0.0: baseline_screening}
for lambda_c in LAMBDA_C_CANDIDATES:
    screening_runs[lambda_c] = fit_cost_policy(
        f"uncertainty_lambda_c_{lambda_c:g}",
        lambda_c=lambda_c,
        seed=SCREENING_SEED,
    )

screening_rows = []
for lambda_c, run in screening_runs.items():
    validation_curve = summarize_curve(
        rollout_curve(run, validation_answers, validation_labels, uncertainty_cost)
    )
    final = validation_curve.loc[validation_curve["step"].eq(MAX_QUESTIONS)].iloc[0]
    screening_rows.append({
        "lambda_c": lambda_c,
        "validation_accuracy": final["accuracy"],
        "validation_macro_f1": final["macro_f1"],
        "validation_mean_cost": final["mean_cost"],
        "validation_mean_sensitive_queries": final["mean_sensitive_queries"],
        "best_epoch": run["best_epoch"],
    })

lambda_sweep_summary = pd.DataFrame(screening_rows).sort_values("lambda_c").reset_index(drop=True)
baseline_accuracy = float(
    lambda_sweep_summary.loc[lambda_sweep_summary["lambda_c"].eq(0.0), "validation_accuracy"].iloc[0]
)
eligible = lambda_sweep_summary[
    lambda_sweep_summary["lambda_c"].gt(0.0)
    & lambda_sweep_summary["validation_accuracy"].ge(
        baseline_accuracy - MAX_VALIDATION_ACCURACY_DROP
    )
]
if eligible.empty:
    selected_row = (
        lambda_sweep_summary[lambda_sweep_summary["lambda_c"].gt(0.0)]
        .sort_values(["validation_accuracy", "validation_mean_cost"], ascending=[False, True])
        .iloc[0]
    )
    selection_rule = "highest validation accuracy; no candidate met the accuracy tolerance"
else:
    selected_row = eligible.sort_values(
        ["validation_mean_cost", "validation_accuracy"], ascending=[True, False]
    ).iloc[0]
    selection_rule = "lowest validation cost within the accuracy tolerance"

COST_WEIGHT = float(selected_row["lambda_c"])
print(f"Selected lambda_c={COST_WEIGHT:g}: {selection_rule}")
display(lambda_sweep_summary)

runs_by_seed = {
    SCREENING_SEED: {
        "baseline": relabel_run(baseline_screening, "baseline"),
        "uncertainty_cost": relabel_run(screening_runs[COST_WEIGHT], "uncertainty_cost"),
    }
}
for seed in SEEDS[1:]:
    runs_by_seed[seed] = {
        "baseline": fit_cost_policy("baseline", lambda_c=0.0, seed=seed),
        "uncertainty_cost": fit_cost_policy(
            f"uncertainty_lambda_c_{COST_WEIGHT:g}",
            lambda_c=COST_WEIGHT,
            seed=seed,
        ),
    }
    runs_by_seed[seed]["uncertainty_cost"] = relabel_run(
        runs_by_seed[seed]["uncertainty_cost"], "uncertainty_cost"
    )

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# Visualization uses the saved full-test trajectories and reports only the
# intended short-query regime. Set this above 5 only for supplementary analyses.
PLOT_MAX_QUESTIONS = 5
saved_curve_path = runs_dir / f"{experiment_name}_test_curves_by_seed.csv"

if saved_curve_path.exists():
    test_curves_by_seed = pd.read_csv(saved_curve_path)
else:
    test_curves_by_seed = pd.concat(
        [
            rollout_curve(run, test_answers, test_labels, uncertainty_cost)
            for seed_runs in runs_by_seed.values()
            for run in seed_runs.values()
        ],
        ignore_index=True,
    )
    test_curves_by_seed.to_csv(saved_curve_path, index=False)

curve_summary = (
    test_curves_by_seed.groupby(["run_name", "step"], as_index=False)
    .agg(
        seeds=("seed", "nunique"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        mean_cost=("mean_cost", "mean"),
        mean_cost_std=("mean_cost", "std"),
        mean_sensitive_queries=("mean_sensitive_queries", "mean"),
    )
    .sort_values(["run_name", "step"])
)
t_critical = float(t.ppf(0.975, len(SEEDS) - 1))
curve_summary["accuracy_ci95"] = (
    t_critical * curve_summary["accuracy_std"] / np.sqrt(curve_summary["seeds"])
)
curve_summary["macro_f1_ci95"] = (
    t_critical * curve_summary["macro_f1_std"] / np.sqrt(curve_summary["seeds"])
)
plot_summary = curve_summary[curve_summary["step"].le(PLOT_MAX_QUESTIONS)].copy()

palette = {"baseline": "#3A6EA5", "uncertainty_cost": "#C44E52"}
labels = {"baseline": "Baseline = uniform cost", "uncertainty_cost": "Uncertainty cost"}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for run_name in ("baseline", "uncertainty_cost"):
    group = plot_summary[plot_summary["run_name"].eq(run_name)]
    color = palette[run_name]
    axes[0].plot(
        group["mean_cost"], group["accuracy_mean"], marker="o", markersize=4,
        color=color, linewidth=2.2, label=labels[run_name],
    )
    axes[0].fill_between(
        group["mean_cost"],
        group["accuracy_mean"] - group["accuracy_ci95"],
        group["accuracy_mean"] + group["accuracy_ci95"],
        color=color, alpha=0.16, linewidth=0,
    )
    axes[1].plot(
        group["step"], group["mean_cost"], marker="o", markersize=4,
        color=color, linewidth=2.2, label=labels[run_name],
    )

axes[0].set(
    xlabel="Mean cumulative uncertainty cost",
    ylabel="Accuracy",
    title=f"Accuracy–cost trade-off (up to {PLOT_MAX_QUESTIONS} queries)",
)
axes[1].set(
    xlabel="Queries asked",
    ylabel="Mean cumulative uncertainty cost",
    title="Cost accumulation in the short-query regime",
    xticks=range(PLOT_MAX_QUESTIONS + 1),
)
for axis in axes:
    axis.grid(alpha=0.22)
    axis.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
fig.tight_layout()
figure_path = paths.figures_root / f"{experiment_name}_cost_tradeoff_q{PLOT_MAX_QUESTIONS}{figure_ext}"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

five_question_summary = plot_summary[plot_summary["step"].eq(PLOT_MAX_QUESTIONS)]
display(five_question_summary)

## Cost-feasible budget sweep

At each step, queries that would exceed the remaining budget are removed from the actor's soft ranking before selecting the highest-ranked feasible query. Confidence stopping is disabled, so the rollout ends only at the query limit or when no query is affordable.

In [ ]:
BUDGET_EVAL_SIZE = len(test_answers)  # lower during development if needed
budget_answers = test_answers[:BUDGET_EVAL_SIZE]
budget_labels = test_labels[:BUDGET_EVAL_SIZE]


@torch.no_grad()
def evaluate_cost_budget(run, budget, answers_cpu=budget_answers, labels_cpu=budget_labels):
    run["actor"].eval()
    run["classifier"].eval()
    predictions, realized_costs, query_counts = [], [], []
    sensitive_counts, exhaustion_flags = [], []

    for start in range(0, len(answers_cpu), config.batch_size):
        answers = answers_cpu[start : start + config.batch_size].to(device)
        mask = torch.zeros_like(answers)
        revealed = torch.zeros_like(answers)
        cumulative_cost = torch.zeros(answers.size(0), device=device)
        queries = torch.zeros(answers.size(0), device=device)
        sensitive_queries = torch.zeros(answers.size(0), device=device)
        active = torch.ones(answers.size(0), dtype=torch.bool, device=device)
        exhausted = torch.zeros_like(active)

        for _ in range(MAX_QUESTIONS):
            feasible = (
                mask.lt(0.5)
                & (cumulative_cost.unsqueeze(1) + uncertainty_cost.unsqueeze(0)).le(float(budget) + 1e-9)
            )
            can_query = active & feasible.any(dim=1)
            exhausted |= active & ~can_query
            if not bool(can_query.any().item()):
                break

            scores = run["actor"](revealed, mask, hard=False)
            scores = scores.masked_fill(~feasible, -torch.inf)
            query_indices = scores.argmax(dim=1)
            query_distribution = F.one_hot(query_indices, num_classes=len(concepts)).float()
            query_distribution *= can_query.unsqueeze(1)

            cumulative_cost += (query_distribution * uncertainty_cost).sum(dim=1)
            queries += can_query.float()
            sensitive_queries += (query_distribution * sensitive_mask).sum(dim=1)
            revealed = apply_query_distribution(revealed, answers, query_distribution)
            mask = torch.clamp(mask + query_distribution, 0.0, 1.0)
            active = can_query

        predictions.extend(run["classifier"](revealed).argmax(dim=1).cpu().tolist())
        realized_costs.extend(cumulative_cost.cpu().tolist())
        query_counts.extend(queries.cpu().tolist())
        sensitive_counts.extend(sensitive_queries.cpu().tolist())
        exhaustion_flags.extend(exhausted.cpu().tolist())

    labels = labels_cpu.tolist()
    return {
        "run_name": run["run_name"],
        "seed": run["seed"],
        "cost_budget": budget,
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "mean_realized_cost": float(np.mean(realized_costs)),
        "mean_queries": float(np.mean(query_counts)),
        "mean_sensitive_queries": float(np.mean(sensitive_counts)),
        "cost_exhaustion_rate": float(np.mean(exhaustion_flags)),
    }


# Saved results are loaded by default because the full-test budget sweep is
# expensive. Enable recomputation only when the experimental setup changes.
RECOMPUTE_BUDGET_SWEEP = False
saved_budget_path = runs_dir / f"{experiment_name}_budget_sweep_by_seed.csv"

if saved_budget_path.exists() and not RECOMPUTE_BUDGET_SWEEP:
    budget_results_by_seed = pd.read_csv(saved_budget_path)
else:
    budget_results_by_seed = pd.DataFrame([
        evaluate_cost_budget(run, budget)
        for seed_runs in runs_by_seed.values()
        for run in seed_runs.values()
        for budget in COST_BUDGETS
    ])
    budget_results_by_seed.to_csv(saved_budget_path, index=False)

budget_results = (
    budget_results_by_seed.groupby(["run_name", "cost_budget"], as_index=False)
    .agg(
        seeds=("seed", "nunique"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        mean_realized_cost=("mean_realized_cost", "mean"),
        mean_queries=("mean_queries", "mean"),
        mean_sensitive_queries=("mean_sensitive_queries", "mean"),
        cost_exhaustion_rate=("cost_exhaustion_rate", "mean"),
    )
)
budget_results["accuracy_ci95"] = (
    t_critical * budget_results["accuracy_std"] / np.sqrt(budget_results["seeds"])
)
display(budget_results)

In [ ]:
screening_history = pd.concat(
    [run["history"] for run in screening_runs.values()], ignore_index=True
)
multi_seed_history = pd.concat(
    [run["history"] for seed_runs in runs_by_seed.values() for run in seed_runs.values()],
    ignore_index=True,
)

output_paths = {
    "cost_vectors": runs_dir / f"{experiment_name}_cost_vectors.csv",
    "lambda_sweep": runs_dir / f"{experiment_name}_lambda_sweep_summary.csv",
    "screening_history": runs_dir / f"{experiment_name}_lambda_sweep_history.csv",
    "multi_seed_history": runs_dir / f"{experiment_name}_history.csv",
    "test_curves_by_seed": runs_dir / f"{experiment_name}_test_curves_by_seed.csv",
    "test_curve_summary": runs_dir / f"{experiment_name}_test_curve_summary.csv",
    "budget_by_seed": runs_dir / f"{experiment_name}_budget_sweep_by_seed.csv",
    "budget_summary": runs_dir / f"{experiment_name}_budget_sweep_summary.csv",
    "selection": runs_dir / f"{experiment_name}_selection.json",
}
cost_table.to_csv(output_paths["cost_vectors"], index=False)
lambda_sweep_summary.to_csv(output_paths["lambda_sweep"], index=False)
screening_history.to_csv(output_paths["screening_history"], index=False)
multi_seed_history.to_csv(output_paths["multi_seed_history"], index=False)
test_curves_by_seed.to_csv(output_paths["test_curves_by_seed"], index=False)
curve_summary.to_csv(output_paths["test_curve_summary"], index=False)
budget_results_by_seed.to_csv(output_paths["budget_by_seed"], index=False)
budget_results.to_csv(output_paths["budget_summary"], index=False)
with open(output_paths["selection"], "w", encoding="utf-8") as handle:
    json.dump(
        {
            "selected_lambda_c": COST_WEIGHT,
            "selection_rule": selection_rule,
            "screening_seed": SCREENING_SEED,
            "seeds": list(SEEDS),
            "calibration_seed": CALIBRATION_SEED,
            "calibration_size": CALIBRATION_SIZE,
            "budget_evaluation_size": BUDGET_EVAL_SIZE,
        },
        handle,
        indent=2,
    )

print(f"Saved figure: {figure_path}")
print("Saved outputs:")
for path in output_paths.values():
    print(path)

## Experimental controls

- The calibration subset is drawn deterministically from the training split and excluded from policy optimization.
- Hyperparameter selection uses validation rollouts only. Test results are produced after selecting `lambda_c`.
- Baseline and uncertainty-cost conditions use identical model architecture, history sampling, designated-query penalty, training depth, and seed set.
- The Concept-QA and CLIP models remain fixed during policy training.
- Uniform query-count cost has zero actor gradient and therefore shares the `lambda_c = 0` baseline; no redundant uniform-cost model is trained.
- Uncertainty markups apply to non-sensitive learned attributes. Designated sensitive attributes retain unit acquisition cost and are controlled separately by `lambda_q`.
- Uncertainty costs are fixed before policy training and used consistently for validation screening, test curves, and cost-feasible budget evaluation.
- Sensitive-query counts are reported separately from acquisition cost.